# Emergent Communication Analysis
This notebook loads the trained checkpoints and compares the emergent language protocols of Gumbel-Softmax and REINFORCE.

In [ ]:

import torch
import numpy as np
import matplotlib.pyplot as plt

import sys
import os
sys.path.insert(0, os.path.abspath('..'))

from data.synthetic_generator import ConceptSpace
from src.sender import Sender
from src.metrics import message_entropy, topographic_similarity, symbol_usage


In [ ]:

# 1. Load Data
VOCAB_SIZES = [6, 6, 8]
CONCEPT_DIM = sum(VOCAB_SIZES)
cs = ConceptSpace(VOCAB_SIZES)
all_concepts_tensor = torch.tensor(cs.all_concepts, dtype=torch.float32)

# Reconstruct all one-hot concepts
one_hots = []
for c in cs.all_concepts:
    row = []
    for val, vs in zip(c, VOCAB_SIZES):
        oh = [0]*vs
        oh[val] = 1
        row.extend(oh)
    one_hots.append(row)
all_concepts = torch.tensor(one_hots, dtype=torch.float32)

# Load Models
def load_sender(ckpt_path):
    ckpt = torch.load(ckpt_path)
    config = ckpt['config']
    sender = Sender(
        concept_dim=CONCEPT_DIM,
        vocab_size=config['model']['vocab_size'],
        embed_dim=config['model']['embed_dim'],
        hidden_dim=config['model']['hidden_dim'],
        max_length=config['model']['max_length'],
        mlp_hidden_dim=config['model']['mlp_hidden_dim']
    )
    sender.load_state_dict(ckpt['sender'])
    sender.eval()
    return sender

sender_gumbel = load_sender('../checkpoints/gumbel_best.pt')
sender_reinforce = load_sender('../checkpoints/reinforce_best.pt')


In [ ]:

# 2. Run Metrics
methods = ['Gumbel-Softmax', 'REINFORCE']
senders = [sender_gumbel, sender_reinforce]

entropies = []
topsims = []

for name, sender in zip(methods, senders):
    ent = message_entropy(sender, all_concepts)
    ts = topographic_similarity(sender, all_concepts)
    entropies.append(ent)
    topsims.append(ts)
    
    print(f'--- {name} ---')
    print(f'Message Entropy: {ent:.4f}')
    print(f'Topographic Similarity: {ts:.4f}')
    
    findings = symbol_usage(sender, all_concepts, VOCAB_SIZES)
    print('Symbol Usage Findings:')
    for f in findings:
        print(f'  - {f}')
    print()


In [ ]:

# 3. Plots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

ax1.bar(methods, topsims, color=['blue', 'orange'])
ax1.set_title('Topographic Similarity')
ax1.set_ylabel('Spearman Correlation')
ax1.set_ylim(-0.1, 1.0)

ax2.bar(methods, entropies, color=['blue', 'orange'])
ax2.set_title('Message Entropy')
ax2.set_ylabel('Bits')

plt.tight_layout()
plt.show()
